# MACD basic trading strategy
Based on the following video\
【A股：这才是MACD的极致用法，我整整读了10遍，太精辟透彻了！-哔哩哔哩】 https://b23.tv/1hVHecU \
Upper bound is defined as the max of the day where 

In [18]:
# Cell 1: Import modules and load/process data
import pandas as pd
import sys
import os
import plotly.graph_objects as go
import pandas_ta as ta

class colors:
       RED = '\033[91m'
       GREEN = '\033[92m'
       YELLOW = '\033[93m'
       BLUE = '\033[94m'
       PINK = '\033[95m'
       CYAN = '\033[96m'
       ENDC = '\033[0m' # Reset to default

# Get the absolute path to the directory containing the current notebook
# and then move up one level to the project root
module_path = os.path.abspath(os.path.join('..'))

if module_path not in sys.path:
       sys.path.append(module_path)

# Now you can import from the modules folder
from modules.data_processor import FeatureFactory
from modules.environment import StockTradingEnv
from modules.model_agent import TradingAgent
from modules.rewards import RewardFunction
# Assuming evaluator.py exists for Module E
from modules.evaluator import Evaluator  # If not present, skip evaluation

# Load raw data
raw_df = pd.read_csv('../data/2454_2015-2025.csv')

# Process data with Module A
factory = FeatureFactory(normalize = False)
df = factory.process(raw_df)
print("Data processed with FeatureFactory.")
print(df.head(5))
df.to_csv('../data/2454_2015-2025_processed.csv', index=False)
print(df.columns)
initial_cash = 10000000
holdings = 0
upper_bound = None
lower_bound = None
PPO_THRESHOLD = 0.5
STOP_LOSS_THRESHOLD = 1
RSI_OVERBOUGHT = 75
RSI_OVERSOLD = 25
asset_snapshot = initial_cash

state = -1 # -1 for no position, 0 for holding cash, 1 for holding stock
for index in range(len(df.index)):
       if (index < 4):
              continue
       center_date = index-2
       three_candles_pos = df.iloc[center_date]['MACDh_12_26_9'] > 0 and df.iloc[center_date -1]['MACDh_12_26_9'] > 0 and df.iloc[center_date +1]['MACDh_12_26_9'] > 0
       three_candles_neg = df.iloc[center_date]['MACDh_12_26_9'] < 0 and df.iloc[center_date -1]['MACDh_12_26_9'] < 0 and df.iloc[center_date +1]['MACDh_12_26_9'] < 0
       if three_candles_pos:
              if ( max(df.iloc[center_date -1 : center_date+2]['MACDh_12_26_9']) == df.iloc[center_date]['MACDh_12_26_9'] and df.iloc[center_date]['PPOh_12_26_9'] > PPO_THRESHOLD ):
                     lower_bound = df.iloc[center_date]['min']
       elif three_candles_neg:
              if ( min(df.iloc[center_date -1 : center_date+2]['MACDh_12_26_9']) == df.iloc[center_date]['MACDh_12_26_9'] and df.iloc[center_date]['PPOh_12_26_9'] < -PPO_THRESHOLD ):
                     upper_bound = df.iloc[center_date]['max']     
       buy_signal = upper_bound is not None and df.iloc[index]['close'] > upper_bound and state != 1
       sell_signal = lower_bound is not None and df.iloc[index]['close'] < lower_bound and state == 1
       rsi = df.iloc[index]['RSI_14']
       rsi_prev = df.iloc[index-1]['RSI_14']
       sell_signal = rsi <= RSI_OVERBOUGHT and rsi_prev > RSI_OVERBOUGHT and state==1 # Sell if RSI is overbought
       sell_signal = sell_signal or (state == 1 and df.iloc[index]['close'] < asset_snapshot * (1 - STOP_LOSS_THRESHOLD)) # Stop loss
       asset = initial_cash + holdings * df.iloc[index]['close']
       
       if sell_signal:
              initial_cash += holdings * df.iloc[index]['close']
              holdings -= holdings
              state = 0
              print(f"{colors.RED}Sell signal at {df.index[index]}: price {df.iloc[index]['close']} crossed below lower bound {lower_bound}{colors.ENDC}")
              print(f"Sell price: {df.iloc[index]['close']}")
              print(f"Holdings after sell: {holdings}, Cash: {initial_cash}")
              print(f"total asset: {initial_cash + holdings * df.iloc[index]['close']}") 
              upper_bound = None
       if buy_signal:
              holdings += initial_cash // df.iloc[index]['close']
              initial_cash -= (initial_cash // df.iloc[index]['close']) * df.iloc[index]['close']
              state = 1
              print(f"{colors.GREEN}Buy signal at {df.index[index]}: price {df.iloc[index]['close']} crossed above upper bound {upper_bound}{colors.ENDC}")
              print(f"Buy price: {df.iloc[index]['close']}")
              print(f"Holdings after buy: {holdings}, Cash: {initial_cash}")
              print(f"total asset: {initial_cash + holdings * df.iloc[index]['close']}")
              lower_bound = None

Data processed with FeatureFactory.
            stock_id  Trading_Volume  ...  signal_buy  signal_sell
date                                  ...                         
2015-05-22      2454         8993585  ...           1            0
2015-05-25      2454         6774725  ...           1            0
2015-05-26      2454         3161817  ...           1            0
2015-05-27      2454         4852855  ...           1            0
2015-05-28      2454         9060437  ...           1            0

[5 rows x 25 columns]
Index(['stock_id', 'Trading_Volume', 'Trading_money', 'open', 'max', 'min',
       'close', 'spread', 'Trading_turnover', 'low', 'high', 'RSI_14',
       'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'SMA_20', 'EMA_26',
       'PPO_12_26_9', 'PPOh_12_26_9', 'PPOs_12_26_9', 'DCL_20_20', 'DCM_20_20',
       'DCU_20_20', 'signal_buy', 'signal_sell'],
      dtype='object')
Buy signal at 2015-08-10 00:00:00: price 288.0 crossed above upper bound 287.5
Buy price: 288.0
